# GraphBG spatially variable genes
- ENV: biopy311

In [1]:
from numpy.random import default_rng
import scanpy as sc
from anndata import AnnData
import scipy
sc.logging.print_header()
from sklearn.metrics.cluster import adjusted_rand_score
import numpy as np
import pandas as pd
import seaborn as sns
import os
import torch
import pandas as pd
from sklearn import metrics
import multiprocessing as mp
from sklearn.metrics.cluster import normalized_mutual_info_score
import pickle
import os.path
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score, homogeneity_score, completeness_score

/home/vanhoan310/miniconda3/envs/gnn/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


scanpy==1.9.3 anndata==0.9.2 umap==0.5.7 numpy==1.24.4 scipy==1.8.1 pandas==1.4.2 scikit-learn==1.3.2 statsmodels==0.14.1 python-igraph==0.11.8 pynndescent==0.5.13


In [2]:
adata = sc.read_h5ad('/data/hoan/spatial_transcriptomics/GraphBG-analysis/Data/DeepGFT/V1_Breast_Cancer_Block_A_Section_1/Section_1_GraphBG.h5ad')

In [3]:
NMI = normalized_mutual_info_score(adata.obs['GraphBG_20'], adata.obs['ground_truth'])
print(NMI)

0.7038061770508631


### SVGs

In [4]:
adata.uns["log1p"] = {"base": None}

In [5]:
import numpy as np
from scipy import sparse

def mean(x, axis=None, dtype=None):
    """Similar to fast_array_utils.stats.mean()."""
    
    if sparse.issparse(x):
        result = x.mean(axis=axis, dtype=dtype)
        return np.asarray(result).squeeze() if axis is not None else result.item()

    # CuPy
    if hasattr(x, "__class__") and "cupy" in str(type(x)).lower():
        return x.mean(axis=axis, dtype=dtype)

    return np.mean(x, axis=axis, dtype=dtype)


import numpy as np
from scipy import sparse

def to_dense(x, to_cpu_memory=False):
    """Similar to fast_array_utils.conv.to_dense()."""

    # SciPy sparse
    if sparse.issparse(x):
        x = x.toarray()

    # PyTorch tensor
    elif hasattr(x, "detach") and hasattr(x, "cpu"):
        x = x.detach()
        if to_cpu_memory:
            x = x.cpu()
        x = x.numpy()

    # CuPy
    elif hasattr(x, "get"):
        if to_cpu_memory:
            x = x.get()

    # Dask
    elif hasattr(x, "compute"):
        x = x.compute()

    else:
        x = np.asarray(x)

    return x

In [6]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from anndata import AnnData
# from fast_array_utils import stats
# from fast_array_utils.conv import to_dense
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
# from scanpy._utils import sanitize_anndata

def spatially_variable_genes(
    adata: AnnData,
    obs_key: str,
    top_k: int,
    cell_size: int,
    min_positive_ratio: float,
    return_list: bool = False,
    show: bool = True,
    **kwargs: int,
):
    """Plot the most spatially variable genes (SVG) for a given `AnnData` object.

    !!! info
        Currently, this function only supports one slide per call.

    Args:
        adata: An `AnnData` object corresponding to one slide.
        obs_key: Key in `adata.obs` that contains the domains. By default, it will use the last available Novae domain key.
        top_k: Number of SVG to be shown.
        cell_size: Size of the cells or spots (`spot_size` argument of `sc.pl.spatial`). By default, it uses the median distance between neighbor cells.
        min_positive_ratio: Genes whose "ratio of cells expressing it" is lower than this threshold are not considered.
        return_list: Whether to return the list of SVG instead of plotting them.
        show: Whether to show the plot.
        **kwargs: Additional arguments for `sc.pl.spatial`.

    Returns:
        A list of SVG names if `return_list` is `True`.
    """
    assert isinstance(adata, AnnData), f"Received adata of type {type(adata)}. Currently only AnnData is supported."

    # obs_key = utils.check_available_domains_key([adata], obs_key)

    sc.tl.rank_genes_groups(adata, groupby=obs_key)
    df = pd.concat(
        [
            sc.get.rank_genes_groups_df(adata, domain).set_index("names")["logfoldchanges"]
            for domain in adata.obs[obs_key].cat.categories
        ],
        axis=1,
    )

    positive_ratio = to_dense(mean(adata.X > 0, axis=0), to_cpu_memory=True)
    valid_vars = adata.var_names[positive_ratio > min_positive_ratio]
    assert len(valid_vars) >= top_k, (
        f"Only {len(valid_vars)} genes are available. Please decrease `top_k` or `min_positive_ratio`."
    )

    svg = df.std(1).loc[valid_vars].sort_values(ascending=False).head(top_k).index

    if return_list:
        return svg.tolist()

    import matplotlib.pyplot as plt
    plt.close("all")  # Close any existing empty figures
    
    # sc.pl.spatial(adata, color=svg, size=1.5, spot_size=cell_size or _get_default_cell_size(adata), show=show, **kwargs)
    sc.pl.spatial(
    adata,
    color=svg,
    size=1.5,
    spot_size=cell_size or _get_default_cell_size(adata),
    ncols=6,       # Increase number of subplots per row
    wspace=0.0001,   # Reduce horizontal spacing
    hspace=0.15,   # Reduce vertical spacing
    show=False,
    **kwargs)
    # plt.savefig(
    # "GraphBG_"+str(top_k)+"_spatially_variable_genes.pdf",
    # format="pdf",
    # bbox_inches="tight")
    plt.savefig(
    "GraphBG_"+str(top_k)+"_spatially_variable_genes.png",
    format="png",
    bbox_inches="tight")
    plt.close()

In [7]:
spatially_variable_genes(adata=adata, obs_key='GraphBG_20', top_k=30, vmax="p95", cell_size=150, min_positive_ratio = 0.6) #0.5

In [8]:
top_k = spatially_variable_genes(adata=adata, obs_key='GraphBG_20', top_k=30, vmax="p95", cell_size=150, min_positive_ratio = 0.6, return_list=True) #0.5

In [9]:
top_k

['CRISP3',
 'MGP',
 'TRBC2',
 'IGLC2',
 'S100G',
 'IGHG3',
 'CPB1',
 'CCDC80',
 'C3',
 'CXCL14',
 'IGHM',
 'IGKC',
 'JCHAIN',
 'IGLC3',
 'IGHG1',
 'SNCG',
 'TIMP1',
 'SERPINA3',
 'MCCD1',
 'IGLC1',
 'CSTA',
 'APOD',
 'WFDC2',
 'TAGLN',
 'LYZ',
 'APOC1',
 'C1R',
 'CORO1A',
 'TFF1',
 'SCGB2A2']